In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from sklearn.metrics import confusion_matrix, classification_report

print("PyTorch Version:", torch.__version__)

In [ ]:
class NpzMedMNIST(Dataset):
    def __init__(self, npz_path, split="train", transform=None):
        data = np.load(npz_path)
        self.images = data[f"{split}_images"]
        self.labels = data[f"{split}_labels"].squeeze()
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img, label = self.images[idx], self.labels[idx]
    
        if img.ndim == 2:  
            img = np.expand_dims(img, -1)
            img = np.repeat(img, 3, axis=-1)
    
        elif img.ndim == 3 and img.shape[-1] == 1:
            img = np.repeat(img, 3, axis=-1)
    
        elif img.ndim == 3 and img.shape[-1] == 3:
            pass  
    
        elif img.ndim == 3 and img.shape[-1] == 9:
            H, W, _ = img.shape
            img = img.reshape(H, W, 3, 3).mean(axis=-1).astype(np.uint8)
    
        else:
            raise ValueError(f"Unexpected image shape {img.shape}")

        img = Image.fromarray(img.astype(np.uint8))
    
        if self.transform:
            img = self.transform(img)
    
        return img, torch.tensor(label, dtype=torch.long)

# Data transforms
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=25),
    #transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10, scale=(0.9, 1.1)),
    #transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    #transforms.RandomGrayscale(p=0.1),
    #transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [ ]:
# Data path - Update this to your Kaggle dataset path
Data_path = "/kaggle/input/datasets/soumyajitgayen/med-mnistbreast-pneumonia-retina-224/breastmnist_224.npz"

# Load datasets
train_dataset = NpzMedMNIST(Data_path, split="train", transform=train_transform)
val_dataset = NpzMedMNIST(Data_path, split="val", transform=val_transform)
test_dataset = NpzMedMNIST(Data_path, split="test", transform=test_transform)

In [ ]:
data = np.load(Data_path)

print(data.files)

In [ ]:
print("Train Images Shape :", data["train_images"].shape)
print("Train Labels Shape :", data["train_labels"].shape)

print("Validation Images Shape :", data["val_images"].shape)
print("Validation Labels Shape :", data["val_labels"].shape)

print("Test Images Shape :", data["test_images"].shape)
print("Test Labels Shape :", data["test_labels"].shape)

In [ ]:
images = data["train_images"]
labels = data["train_labels"]

plt.figure(figsize=(12,6))

for i in range(8):
    
    plt.subplot(2,4,i+1)
    
    plt.imshow(images[i])
    plt.title(f"Label : {labels[i][0]}")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
class CNN_1(nn.Module):

    def __init__(self, num_classes=2):
        super(CNN_1, self).__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(32 * 112 * 112, 128),
            nn.ReLU(),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        x = self.features(x)
        x = self.classifier(x)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device :", device)

model = CNN_1(num_classes=2).to(device)

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(model.parameters(),lr=0.001)

In [ ]:
def evaluate(model, loader):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():

        for images, labels in progress_bar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

            current_acc = 100 * correct / total

            progress_bar.set_postfix(
                loss=loss.item(),
                acc=current_acc
            )

    epoch_loss = running_loss / len(loader)

    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, loader):

    model.eval()

    running_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(
        loader,
        desc="Validation",
        leave=False
    )

    with torch.no_grad():

        for images, labels in progress_bar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()

            current_acc = 100 * correct / total

            progress_bar.set_postfix(
                loss=loss.item(),
                acc=current_acc
            )

    epoch_loss = running_loss / len(loader)

    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
num_epochs = 10

train_losses = []
val_losses = []

train_accs = []
val_accs = []


for epoch in range(num_epochs):

    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}%")

    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.2f}%")

    print("-"*50)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_accs, label="Train Accuracy")
plt.plot(val_accs, label="Validation Accuracy")

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.title("Training vs Validation Accuracy")

plt.legend()

plt.show()

In [ ]:
test_loss, test_acc = evaluate(model, test_loader)

print(f"Test Accuracy : {test_acc:.2f}%")

In [ ]:
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    
    for images, labels in test_loader:
        
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())


cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(8,6))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.show()

In [ ]:
print(classification_report(all_labels, all_preds))

In [ ]:
torch.save(model.state_dict(), "bloodmnist_cnn.pth")

print("Model Saved Successfully!")